In [1]:
import pandas as pd
import numpy as np
import optuna
import dagshub
import mlflow

from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.compose import ColumnTransformer,TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor,StackingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler,OneHotEncoder,OrdinalEncoder,PowerTransformer
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error,r2_score

c:\Users\Jay Kanakia\Desktop\CampusX\Projects\swiggy_delivery_time_prediction\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv(r'C:\Users\Jay Kanakia\Desktop\CampusX\Projects\swiggy_delivery_time_prediction\notebooks\data_eda.csv')
df.sample(2)

,rider_id,age,ratings,restaurant_latitude,restaurant_longitude,delivery_latitude,delivery_longitude,order_date,weather,traffic,...,city_name,order_day,order_month,order_day_of_week,is_weekend,pickup_time_minutes,order_time_hour,order_time_of_day,distance,distance_type
36435,GOARES13DEL02,33.0,4.8,15.506205,73.766668,15.556205,73.816668,2022-02-11,sandstorms,medium,...,GOA,11,2,friday,0,15.0,17.0,afternoon,7.720450,medium
20524,JAPRES03DEL01,23.0,4.6,26.913483,75.803139,26.983483,75.873139,2022-03-23,sandstorms,jam,...,JAP,23,3,wednesday,0,10.0,20.0,evening,10.427236,long


In [3]:
# dropping columns
df.drop(columns=['rider_id',
                    'restaurant_latitude',
                    'restaurant_longitude',
                    'delivery_latitude',
                    'delivery_longitude',
                    'order_date',
                    "order_time_hour",
                    "order_day",
                    "city_name",
                    "order_day_of_week",
                    "order_month"],inplace=True)

df.sample(2)

,age,ratings,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,time_taken,is_weekend,pickup_time_minutes,order_time_of_day,distance,distance_type
30093,36.0,4.3,windy,high,1,meal,scooter,0.0,no,metropolitian,31,0,15.0,afternoon,5.992276,medium
10981,39.0,4.7,stormy,low,1,buffet,scooter,1.0,no,metropolitian,29,1,5.0,night,10.683837,long


In [4]:
df.isnull().sum()

age                    1854
ratings                1908
weather                 525
traffic                 510
vehicle_condition         0
type_of_order             0
type_of_vehicle           0
multiple_deliveries     993
festival                228
city_type              1198
time_taken                0
is_weekend                0
pickup_time_minutes    1640
order_time_of_day      2070
distance               3630
distance_type          3630
dtype: int64

In [5]:
df.duplicated().isnull().sum()

np.int64(0)

In [6]:
missing_col = df.isnull().any(axis=0).loc[lambda x: x].index
missing_col

Index(['age', 'ratings', 'weather', 'traffic', 'multiple_deliveries',
       'festival', 'city_type', 'pickup_time_minutes', 'order_time_of_day',
       'distance', 'distance_type'],
      dtype='object')

## Stacking Regressor

In [7]:
temp_df = df.copy().dropna()
temp_df.sample(2)

,age,ratings,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,time_taken,is_weekend,pickup_time_minutes,order_time_of_day,distance,distance_type
7702,29.0,4.5,cloudy,medium,2,meal,electric_scooter,0.0,no,urban,18,0,5.0,evening,7.448561,medium
41084,35.0,4.0,fog,low,0,snack,motorcycle,2.0,no,metropolitian,43,0,5.0,night,10.867421,long


In [8]:
X = df.drop(columns=['time_taken'])
y = df['time_taken']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

X_train.shape,y_test.shape

((36401, 15), (9101,))

In [9]:
temp_df.isnull().sum()

age                    0
ratings                0
weather                0
traffic                0
vehicle_condition      0
type_of_order          0
type_of_vehicle        0
multiple_deliveries    0
festival               0
city_type              0
time_taken             0
is_weekend             0
pickup_time_minutes    0
order_time_of_day      0
distance               0
distance_type          0
dtype: int64

In [10]:
# transforming target column

pt = PowerTransformer(method='yeo-johnson')

y_train_pt = pt.fit_transform(y_train.values.reshape(-1,1))
y_test_pt = pt.transform(y_test.values.reshape(-1,1))

y_train_pt

array([[-0.73874627],
       [-0.60998789],
       [ 0.39299837],
       ...,
       [ 0.77215502],
       [ 0.39299837],
       [ 0.49090732]], shape=(36401, 1))

In [11]:
# preprocessing

num_cols = ["age","ratings","pickup_time_minutes","distance"]

nominal_cat_cols = ['weather',
                    'type_of_order',
                    'type_of_vehicle',
                    "festival",
                    "city_type",
                    "is_weekend",
                    "order_time_of_day"]

ordinal_cat_cols = ["traffic","distance_type"]

In [12]:
# generate order for ordinal encoding

traffic_order = ["low","medium","high","jam"]

distance_type_order = ["short","medium","long","very_long"]

In [13]:
# build a preprocessor

preprocessor = ColumnTransformer(transformers=[
    ("scale", MinMaxScaler(), num_cols),
    ("nominal_encode", OneHotEncoder(drop="first",handle_unknown="ignore",
                                     sparse_output=False), nominal_cat_cols),
    ("ordinal_encode", OrdinalEncoder(categories=[traffic_order,distance_type_order],
                                      encoded_missing_value=-999,
                                      handle_unknown="use_encoded_value",
                                      unknown_value=-1), ordinal_cat_cols)
],remainder="passthrough",n_jobs=-1,force_int_remainder_cols=False,verbose_feature_names_out=False)


preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('scale', ...), ('nominal_encode', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",-1
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and `

In [14]:
pipeline = Pipeline(
    [
        ('preprocessor',preprocessor)
    ]
)

In [15]:
X_train_trans = pipeline.fit_transform(X_train)
X_test_trans = pipeline.transform(X_test)

c:\Users\Jay Kanakia\Desktop\CampusX\Projects\swiggy_delivery_time_prediction\myenv\Lib\site-packages\sklearn\compose\_column_transformer.py:978: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(


## Optuna

In [16]:
dagshub.init(repo_owner='jay-kanakia', repo_name='swiggy_delivery_time_prediction', mlflow=True)
mlflow.set_tracking_uri('https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow')
mlflow.set_experiment(experiment_name='Exp 5 : Stacking Regressor')

Accessing as jay-kanakia

Initialized MLflow to track repo "jay-kanakia/swiggy_delivery_time_prediction"

Repository jay-kanakia/swiggy_delivery_time_prediction initialized!

<Experiment: artifact_location='mlflow-artifacts:/5c98cdecf5564a139a53aa054ee60eb3', creation_time=1775489302047, experiment_id='6', last_update_time=1775489302047, lifecycle_stage='active', name='Exp 5 : Stacking Regressor', tags={'mlflow.experimentKind': 'custom_model_development'}, workspace='default'>

In [17]:
best_lgbm_params = {
        'n_estimator': 40,
        'max_depth': 11,
        'learning_rate': 0.18024722308848545,
        'subsample': 0.7972018508352986,
        'num_leaves': 118,
        'min_child_weight': 14,
        'min_split_gain': 0.007311476316166993,
        'reg_lambda': 54.73782637648871
                    }

best_rf_params = {
        'n_estimators': 424,
        'max_depth': 28,
        'max_features': None,
        'min_samples_split': 8,
        'min_samples_leaf': 6,
        'max_samples': 0.8542421813553751
        }

best_lgbm = LGBMRegressor(**best_lgbm_params)
best_rf = RandomForestRegressor(**best_rf_params)

In [18]:
def objective(trial):

    with mlflow.start_run(nested=True):
        meta_model_name = trial.suggest_categorical('meta_model_name',['LR','KNN','DT'])

        if meta_model_name == 'LR':
            meta = LinearRegression()

        elif meta_model_name == 'KNN':
            params = {
                'n_neighbors' : trial.suggest_int('n_neighbors',1,7),
                'weights' : trial.suggest_categorical('weights',['uniform','distance']),
                'n_jobs' : -1
            }
            meta = KNeighborsRegressor(**params)

        elif meta_model_name == 'DT':
            params = {
                'max_depth' : trial.suggest_int('max_depth',1,10),
                'min_samples_split' : trial.suggest_int('min_samples_split',2,10),
                'min_samples_leaf' : trial.suggest_int('min_samples_leaf',1,10),
                'random_state' : 42
            }
            meta  = DecisionTreeRegressor(**params)


        # log meta model name
        mlflow.log_param('meta_model_name',meta_model_name)

        #Stacking Regressor

        srg = StackingRegressor(
            estimators=[('rf',best_rf),
                        ('lgbm',best_lgbm)],
                        final_estimator=meta,
                        cv=5, n_jobs=-1
        )

        model = TransformedTargetRegressor(regressor=srg,transformer=pt)

        # train the model
        # score = cross_val_score(model,X_train_trans,y_train,cv=5,scoring='neg_mean_absolute_error',n_jobs=-1)
        # mean_score = score.mean()
        model.fit(X_train_trans,y_train)

        # get the prediction
        y_pred_train = model.predict(X_train_trans)
        y_pred_test = model.predict(X_test_trans)
        error = mean_absolute_error(y_test,y_pred_test)

        # logging errors
        mlflow.log_metric('training_error',mean_absolute_error(y_train,y_pred_train))
        mlflow.log_metric('testing_error',mean_absolute_error(y_test,y_pred_test))
        mlflow.log_metric('training_accuracy',r2_score(y_train,y_pred_train))
        mlflow.log_metric('testing_accuracy',r2_score(y_test,y_pred_test))
        #mlflow.log_metric('cv_mean_absolute_error',mean_score)

        return error

In [ ]:
study = optuna.create_study(direction='minimize',study_name='Stacking regressor')

with mlflow.start_run(run_name='best_model'):

    study.optimize(objective,n_trials=20,n_jobs=-1,show_progress_bar=True)

    #log best params
    mlflow.log_params(study.best_params)

    # log best score
    mlflow.log_param('best_score',study.best_value)

[I 2026-04-07 09:51:59,323] A new study created in memory with name: Stacking regressor
  0%|          | 0/20 [00:00<?, ?it/s]c:\Users\Jay Kanakia\Desktop\CampusX\Projects\swiggy_delivery_time_prediction\myenv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
